In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gurobipy as gp
from gurobipy import GRB

In [28]:
model = gp.Model("Reverse_Logistics")

# ==========================================================
# SETS (Initially Empty - Populate later from datasets)
# ==========================================================
collection = pd.read_csv("collection_centers.csv")
dismantling = pd.read_csv("dismantling_centers.csv")
recycling = pd.read_csv("recycling_centers.csv")
dist_DR_df = pd.read_csv("Dismantling_to_Recycling_Distances.csv")
dist_CD_df = pd.read_csv("Collection_to_Dismantling_Distances.csv")

# Collection Centres
C = collection["Name"].tolist()      # Collection centers

# Dismantling Centres
D = dismantling["Name"].tolist()     # Dismantling centers

# Recycling Centres
R = recycling["Name"].tolist()       # Recycling centers

# Optional: dictionaries of coordinates
collection_coord = dict(zip(C, zip(collection["Latitude"], collection["Longitude"])))
dismantling_coord = dict(zip(D, zip(dismantling["Latitude"], dismantling["Longitude"])))
recycling_coord = dict(zip(R, zip(recycling["Latitude"], recycling["Longitude"])))

# Print to verify
print("Collection centers (C):", C)
print("Dismantling centers (D):", D)
print("Recycling centers (R):", R)
# # (Optional) Time periods, since Mt is indexed by t in the paper
# T = []


Collection centers (C): ['Berlin', 'Hamburg', 'Munich', 'Cologne', 'Amsterdam', 'Rotterdam', 'The Hague', 'Utrecht', 'Paris', 'Lyon', 'Marseille', 'Antwerp', 'Brussels', 'Vienna', 'Stockholm', 'Barcelona', 'Madrid', 'Prague', 'Copenhagen', 'Aalborg', 'Warsaw', 'Lublin', 'Helsinki', 'Lisbon', 'Porto', 'Braga', 'Setubal']
Dismantling centers (D): ['Duisburg', 'Rotterdam', 'Antwerp', 'Hamburg', 'Lyon', 'Frankfurt', 'Gdańsk', 'Le Havre', 'Marseille', 'Vienna', 'Milan', 'Katowice']
Recycling centers (R): ['Lacq', 'Grenoble', 'Rotterdam', 'Stuttgart', 'Darmstadt', 'Antwerp', 'Hamburg']


In [29]:
# PARAMETERS
# -----------------------------
# Fixed Costs
# -----------------------------
FC_c = 200000      # Fixed cost of opening collection centre
FC_d = 150000      # Fixed cost of opening dismantling centre
FC_r = 100000      # Fixed cost of opening recycling centre

# -----------------------------
# Maximum Capacities
# -----------------------------
Cap_c_max = 4000    
Cap_d_max = 4000
Cap_r_max = 1000

# -----------------------------
# Distances
# -----------------------------
# Create dictionaries
dist_CD = {
    (row["Collection"], row["Dismantling"]): row["Distance_km"]
    for _, row in dist_CD_df.iterrows()
}

dist_DR = {
    (row["Dismantling"], row["Recycling"]): row["Distance_km"]
    for _, row in dist_DR_df.iterrows()
}

# -----------------------------
# Transportation Cost
# -----------------------------
TransCost = 0.395

# -----------------------------
# Conversion Factors
# -----------------------------
alpha = 0.1
beta = 0.95

# -----------------------------
# Supply
# -----------------------------
S_c = dict(zip(collection["Name"], collection["Supply"]))

# -----------------------------
# Magnet Demand
# -----------------------------
Mt = 5000         # indexed by time period

# -----------------------------
# Emission Factor
# -----------------------------
Em = 0.046

# -----------------------------
# Capacity Cost
# -----------------------------
TotalCapCost = 25

#Percentage of recycled material that can be used in the production of new magnets
gamma = 0.30
# fr = 0  # Initialize fr to zero

In [30]:
# DECISION VARIABLES
# Facility opening variables
# Yc = model.addVars(C, vtype=GRB.BINARY, name="Yc")
Yd = model.addVars(D, vtype=GRB.BINARY, name="Yd")
Yr = model.addVars(R, vtype=GRB.BINARY, name="Yr")

# Material flow variables
Fcd = model.addVars(C, D, vtype=GRB.CONTINUOUS, name="Fcd")
Fdr = model.addVars(D, R, vtype=GRB.CONTINUOUS, name="Fdr")

# # Recycled material
# fr = model.addVars(R, vtype=GRB.CONTINUOUS, name="fr")

# Capacity variables
# Cap_c = model.addVars(C, vtype=GRB.CONTINUOUS, name="Cap_c")
Cap_d = model.addVars(D, vtype=GRB.CONTINUOUS, ub=Cap_d_max, name="Cap_d")
Cap_r = model.addVars(R, vtype=GRB.CONTINUOUS, ub=Cap_r_max, name="Cap_r")

# Linearization variables
# z_c = model.addVars(C, vtype=GRB.CONTINUOUS, name="z_c")
z_d = model.addVars(D, vtype=GRB.CONTINUOUS, name="z_d")
z_r = model.addVars(R, vtype=GRB.CONTINUOUS, name="z_r")

In [31]:
#Objective Function
# ----------------------------------------------------------
model.setObjective((gp.quicksum(Fcd[c,d]*dist_CD[c,d] for c in C for d in D) + 
                   gp.quicksum(Fdr[d,r]*dist_DR[d,r] for d in D for r in R))*TransCost + 
                  #  gp.quicksum(Yc[c] for c in C)*FC_c +
                   gp.quicksum(Yr[r] for r in R)*FC_r + 
                   gp.quicksum(Yd[d] for d in D)*FC_d +
                   TotalCapCost*(gp.quicksum(Cap_d[d] for d in D)+gp.quicksum(Cap_r[r] for r in R)) +
                   Em*(gp.quicksum(dist_CD[c,d] for c in C for d in D)+gp.quicksum(dist_DR[d,r] for d in D for r in R))
                   ,GRB.MINIMIZE)

In [32]:
# ==========================================================
# CONSTRAINTS
# ==========================================================

# ----------------------------------------------------------
# (2) Collection Centre Linearization
# zc <= Capc_max * Yc
# ----------------------------------------------------------
# model.addConstrs(
#     (z_c[c] <= Cap_c_max * Yc[c] for c in C),
#     name="CollectionLinearization1"
# )

# ----------------------------------------------------------
# (3) Total Motor Supply
# S_c >= Σ Σ Fcd
# ----------------------------------------------------------
for c in C:
    model.addConstr(
        gp.quicksum(Fcd[c, d] for d in D) <= S_c[c],
        name=f"MotorSupply_{c}"
    )

# ----------------------------------------------------------
# (4) zc <= Capc
# ----------------------------------------------------------
# model.addConstrs(
#     (z_c[c] <= Cap_c[c] for c in C),
#     name="CollectionCapacity1"
# )

# ----------------------------------------------------------
# (5) zc >= Capc - Capc_max*(1-Yc)
# ----------------------------------------------------------
# model.addConstrs(
#     (
#         z_c[c] >= Cap_c[c] - Cap_c_max * (1 - Yc[c])
#         for c in C
#     ),
#     name="CollectionCapacity2"
# )

# ----------------------------------------------------------
# (6) Σ Fcd <= zc
# ----------------------------------------------------------
# model.addConstrs(
#     (
#         gp.quicksum(Fcd[c, d] for d in D) <= z_c[c]
#         for c in C
#     ),
#     name="CollectionFlow"
# )

#Constraints for dismantling centers
#

model.addConstrs(gp.quicksum(Fcd[c,d] for c in C)<= z_d[d] for d in D)
# ==========================================================
# Dismantling Centre Constraints
# ==========================================================

# ----------------------------------------------------------
# (7) zd <= Capd
# ----------------------------------------------------------
model.addConstrs(
    (z_d[d] <= Cap_d[d] for d in D),
    name="DismantlingCapacity1"
)

# ----------------------------------------------------------
# (8) zd >= Capd - Capd_max*(1-Yd)
# ----------------------------------------------------------
model.addConstrs(
    (
        z_d[d] >= Cap_d[d] - Cap_d_max * (1 - Yd[d])
        for d in D
    ),
    name="DismantlingCapacity2"
)

# ----------------------------------------------------------
# (9) Σ Fdr <= zd
# ----------------------------------------------------------
model.addConstrs(
    (
        gp.quicksum(Fdr[d, r] for r in R) <= z_d[d]
        for d in D
    ),
    name="DismantlingFlow"
)
# ----------------------------------------------------------
# (9) Cap_d_max*Yd => zd
# ----------------------------------------------------------
model.addConstrs(z_d[d] <= Cap_d_max * Yd[d] for d in D)

# ==========================================================
# Recycling Centre Constraints
# ==========================================================

# ----------------------------------------------------------
# (10) zr <= Capr
# ----------------------------------------------------------
model.addConstrs(
    (z_r[r] <= Cap_r[r] for r in R),
    name="RecyclingCapacity1"
)

# ----------------------------------------------------------
# (11) zr >= Capr - Capr_max*(1-Yr)
# ----------------------------------------------------------
model.addConstrs(
    (
        z_r[r] >= Cap_r[r] - Cap_r_max * (1 - Yr[r])
        for r in R
    ),
    name="RecyclingCapacity2"
)

# ----------------------------------------------------------
# (12) Incoming flow <= recycling capacity
# Σ_d Fdr <= zr
# ----------------------------------------------------------
model.addConstrs(
    (
        gp.quicksum(Fdr[d, r] for d in D) <= z_r[r]
        for r in R
    ),
    name="RecyclingFlow"
)

# ----------------------------------------------------------
# (9) Cap_r_max*Yr => z_r
# ----------------------------------------------------------
model.addConstrs(z_r[r] <= Cap_r_max * Yr[r] for r in R)
# ==========================================================
# ==========================================================
# Material Balance Constraints
# ==========================================================

# ----------------------------------------------------------
# (13)
# Σ Fdr = α Σ Fcd
# ----------------------------------------------------------
# model.addConstr(
#     gp.quicksum(Fdr[d, r] for d in D for r in R)
#     ==
#     alpha * gp.quicksum(Fcd[c, d] for c in C for d in D),
#     name="ConversionToParts"
# )
for d in D:
    model.addConstr(
        gp.quicksum(Fdr[d,r] for r in R)
        ==
        alpha * gp.quicksum(Fcd[c,d] for c in C)
    )
# ----------------------------------------------------------
# (14)
# fr = β Σ Fdr
#
# Since fr is indexed by recycling centre, a better formulation
# is to define it per recycling centre.
# ----------------------------------------------------------
model.addConstr(
    (
        beta*gp.quicksum(Fdr[d, r] for d in D for r in R) >= 0.25*Mt
    ),
    name="RecoveredMaterial"
)

# ----------------------------------------------------------
# (15)
# fr >= Mt
# If Mt is time dependent, replace Mt with Mt[t].
# Here assuming one demand value.
# ----------------------------------------------------------
# model.addConstr(
#     (fr >= Mt ),
#     name="Demand"
# )

# ==========================================================
# Facility Activation Constraints
# ==========================================================

# ----------------------------------------------------------
# (16)
# At least one collection centre
# ----------------------------------------------------------
# model.addConstr(
#     gp.quicksum(Yc[c] for c in C) >= 1,
#     name="OpenCollection"
# )

# ----------------------------------------------------------
# (17)
# At least one dismantling centre
# ----------------------------------------------------------
model.addConstr(
    gp.quicksum(Yd[d] for d in D) >= 1,
    name="OpenDismantling"
)

# ----------------------------------------------------------
# (18)
# At least one recycling centre
# ----------------------------------------------------------
model.addConstr(
    gp.quicksum(Yr[r] for r in R) >= 1,
    name="OpenRecycling"
)

<gurobi.Constr *Awaiting Model Update*>

In [33]:
model.update()
model.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-12450HX, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 130 rows, 465 columns and 1491 nonzeros (Min)
Model fingerprint: 0x6d140304
Model has 437 linear objective coefficients and an objective constant of 16666.5314
Variable types: 446 continuous, 19 integer (19 binary)
Coefficient statistics:
  Matrix range     [1e-01, 4e+03]
  Objective range  [8e+00, 2e+05]
  Bounds range     [1e+00, 4e+03]
  RHS range        [1e+00, 4e+03]

Presolve removed 57 rows and 19 columns
Presolve time: 0.00s
Presolved: 73 rows, 446 columns, 1389 nonzeros
Variable types: 427 continuous, 19 integer (19 binary)

Root relaxation: objective 1.959349e+06, 77 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntIn

In [34]:
print(f"\nOptimal Objective Value: {model.ObjVal:,.2f}")


Optimal Objective Value: 2,635,174.10


In [35]:
print("\n========== COLLECTION CENTRES OPENED ==========")

# for c in C:
#     if Yc[c].X > 0.5:
#         print(f"{c}")


========== COLLECTION CENTRES OPENED ==========


In [36]:
print("\n========== DISMANTLING CENTRES OPENED ==========")

for d in D:
    if Yd[d].X > 0.5:
        print(f"{d}")


========== DISMANTLING CENTRES OPENED ==========
Antwerp
Hamburg
Marseille
Vienna


In [37]:
print("\n========== RECYCLING CENTRES OPENED ==========")

for r in R:
    if Yr[r].X > 0.5:
        print(f"{r}")


========== RECYCLING CENTRES OPENED ==========
Antwerp
Hamburg


In [38]:
print("\n========== COLLECTION → DISMANTLING ==========")

for c in C:
    for d in D:
        if Fcd[c, d].X > 1e-6:
            print(f"{c:15s} --> {d:15s} : {Fcd[c,d].X:8.2f}")


========== COLLECTION → DISMANTLING ==========
Berlin          --> Hamburg         :  1238.00
Hamburg         --> Hamburg         :  1238.00
Munich          --> Vienna          :  1238.00
Cologne         --> Antwerp         :  1238.00
Amsterdam       --> Antwerp         :   271.00
Rotterdam       --> Antwerp         :   271.00
The Hague       --> Antwerp         :   271.00
Utrecht         --> Antwerp         :   271.00
Paris           --> Antwerp         :   879.00
Lyon            --> Marseille       :   879.00
Marseille       --> Marseille       :   879.00
Antwerp         --> Antwerp         :   320.00
Brussels        --> Antwerp         :   320.00
Vienna          --> Vienna          :   488.00
Stockholm       --> Hamburg         :   536.00
Barcelona       --> Marseille       :   769.00
Madrid          --> Marseille       :   132.89
Prague          --> Vienna          :   579.00
Copenhagen      --> Hamburg         :   228.00
Aalborg         --> Hamburg         :   228.00
Warsaw      

In [39]:
print("\n========== DISMANTLING → RECYCLING ==========")

for d in D:
    for r in R:
        if Fdr[d, r].X > 1e-6:
            print(f"{d:15s} --> {r:15s} : {Fdr[d,r].X:8.2f}")


========== DISMANTLING → RECYCLING ==========
Antwerp         --> Antwerp         :   384.10
Hamburg         --> Hamburg         :   346.80
Marseille       --> Antwerp         :   265.99
Vienna          --> Hamburg         :   318.90


In [40]:
# print("\n========== COLLECTION CENTRES ==========")

# for c in C:
#     if Yc[c].X > 0.5:      # Only opened centres

#         flow = sum(Fcd[c,d].X for d in D)
#         installed_capacity = Cap_c[c].X
#         utilized_capacity = z_c[c].X

#         if utilized_capacity > 1e-6:
#             utilization = 100 * flow / utilized_capacity
#         else:
#             utilization = 0

#         print(f"{c}")
#         print(f"  Flow              : {flow:.2f}")
#         print(f"  Installed Capacity: {installed_capacity:.2f}")
#         print(f"  Effective Capacity: {utilized_capacity:.2f}")
#         print(f"  Utilization       : {utilization:.1f}%")
        # print()

In [41]:
print("\n========== DISMANTLING CENTRES ==========")

for d in D:
    if Yd[d].X > 0.5:

        incoming = sum(Fcd[c,d].X for c in C)
        outgoing = sum(Fdr[d,r].X for r in R)

        installed_capacity = Cap_d[d].X
        utilized_capacity = z_d[d].X

        if utilized_capacity > 1e-6:
            utilization = 100 * outgoing / utilized_capacity
        else:
            utilization = 0

        print(f"{d}")
        print(f"  Incoming Motors   : {incoming:.2f}")
        print(f"  Outgoing Parts    : {outgoing:.2f}")
        print(f"  Installed Capacity: {installed_capacity:.2f}")
        print(f"  Effective Capacity: {utilized_capacity:.2f}")
        print(f"  Utilization       : {utilization:.1f}%")
        print()


========== DISMANTLING CENTRES ==========
Antwerp
  Incoming Motors   : 3841.00
  Outgoing Parts    : 384.10
  Installed Capacity: 3841.00
  Effective Capacity: 3841.00
  Utilization       : 10.0%

Hamburg
  Incoming Motors   : 3468.00
  Outgoing Parts    : 346.80
  Installed Capacity: 3468.00
  Effective Capacity: 3468.00
  Utilization       : 10.0%

Marseille
  Incoming Motors   : 2659.89
  Outgoing Parts    : 265.99
  Installed Capacity: 2659.89
  Effective Capacity: 2659.89
  Utilization       : 10.0%

Vienna
  Incoming Motors   : 3189.00
  Outgoing Parts    : 318.90
  Installed Capacity: 3189.00
  Effective Capacity: 3189.00
  Utilization       : 10.0%



In [42]:
print("\n========== RECYCLING CENTRES ==========")

for r in R:
    if Yr[r].X > 0.5:

        incoming = sum(Fdr[d,r].X for d in D)

        installed_capacity = Cap_r[r].X
        utilized_capacity = z_r[r].X

        if utilized_capacity > 1e-6:
            utilization = 100 * incoming / utilized_capacity
        else:
            utilization = 0

        print(f"{r}")
        print(f"  Incoming Material : {incoming:.2f}")
        print(f"  Installed Capacity: {installed_capacity:.2f}")
        print(f"  Effective Capacity: {utilization:.2f}")
        print(f"  Utilization       : {utilization:.1f}%")
        print()


========== RECYCLING CENTRES ==========
Antwerp
  Incoming Material : 650.09
  Installed Capacity: 650.09
  Effective Capacity: 100.00
  Utilization       : 100.0%

Hamburg
  Incoming Material : 665.70
  Installed Capacity: 665.70
  Effective Capacity: 100.00
  Utilization       : 100.0%



In [43]:
if model.status == GRB.INFEASIBLE:
    model.computeIIS()
    model.write("model_infeasibility.ilp")
    print("IIS written to model_infeasibility.ilp")